In [2]:
import numpy as np
import joblib
from utils import train_categorical_model, prepare_datasets, validate_results, train_incident_agent, generate_oof_features, find_optimal_threshold
from rcf_model import RCF
from meta_scorer import train_fusion_meta_learner, CostSensitiveMetaLearner, _incident_entropy
from catboost import CatBoostClassifier
from sklearn.metrics import confusion_matrix, classification_report
from soar_zta import ZeroTrustSOARAgent
from playbook_editor import PlaybookEditorAgent

In [2]:
file_path = r"UNSW-NB15\unsw_train.csv"

# FIX (PCA Leakage): prepare_datasets now returns X_num_raw (6th value) —
# the raw, unscaled numerical frame. This is passed to generate_oof_features
# so each fold fits its own scaler+PCA exclusively on its training split.
# X_train_num (PCA-transformed using the full-train artifact) is kept for
# the final RCF training step in cell 5.
X_train_cat, X_train_num, X_train_num_raw, y_bin, cat_cols, y_multi = prepare_datasets(file_path, is_train=True)

# FIX (PCA Leakage): Pass X_train_num_raw instead of X_train_num.
# generate_oof_features will fit a fresh scaler+PCA per fold internally.

# FIX: Train the final RCF model FIRST to establish the global mathematical anchors
print("\n--- Establishing Global RCF Anchors ---")
final_rcf = RCF(num_trees=40, tree_size=256)
y_bin_reset = y_bin.reset_index(drop=True)
X_train_num_normal_full = X_train_num[y_bin_reset == 0]

# X_train_num is the full-train PCA output from prepare_datasets — correct input
# for the final RCF. X_train_num_raw is NOT used here because the saved PCA
# artifact must be consistent with what predict_proba receives at test time.
final_rcf.fit_predict(X_train_num_normal_full)
global_p1 = final_rcf._score_p1
global_p99 = final_rcf._score_p99

final_rcf.save_model(r"Saves/rcf_base.pkl")

oof_cat_scores, oof_rcf_scores, oof_incident_entropy = generate_oof_features(
    X_train_cat, X_train_num_raw, y_bin, cat_cols,
    train_cat_func=train_categorical_model,
    rcf_class=RCF,
    train_incident_func=train_incident_agent,
    n_splits=5,
    y_multi=y_multi,
    global_p1=global_p1,
    global_p99=global_p99
)

Loading and transforming data from UNSW-NB15\unsw_train.csv (Mode: Train)...
Final Categorical features: 6
Numerical features processed: 20 Principal Components

--- Establishing Global RCF Anchors ---
Building RRCF Forest (40 trees)...


RRCF Training & Scoring: 100%|██████████| 37000/37000 [06:27<00:00, 95.38it/s] 


[RCF] Calibration anchors — p1=2.4210  p99=62.4926  effective_ceiling=93.7389  n_samples=37000
RCF state successfully saved to Saves/rcf_base.pkl

Generating 5-Fold OOF Predictions for Meta-Learner training...
  -> Processing Fold 1/5...

Training CatBoost (with High Regularization)...
Building RRCF Forest (40 trees)...


RRCF Training & Scoring: 100%|██████████| 29600/29600 [05:14<00:00, 94.07it/s] 


[RCF] Calibration anchors — p1=2.4210  p99=62.4926  effective_ceiling=93.7389  n_samples=29600


...


[RCF] Score distribution — min=0.0000  mean=0.2869  max=1.0000  % at ceiling (≥0.999): 2.7%

Training Incident Agent (Multi-class Threat Classifier)...
  -> Processing Fold 5/5...

Training CatBoost (with High Regularization)...
Building RRCF Forest (40 trees)...


RRCF Training & Scoring: 100%|██████████| 29600/29600 [05:01<00:00, 98.11it/s] 


[RCF] Calibration anchors — p1=2.4210  p99=62.4926  effective_ceiling=93.7389  n_samples=29600


RRCF Blind Test Scoring: 100%|██████████| 16466/16466 [02:46<00:00, 99.17it/s] 



[RCF] Score distribution — min=0.0000  mean=0.3321  max=1.0000  % at ceiling (≥0.999): 5.7%

Training Incident Agent (Multi-class Threat Classifier)...
OOF Generation Complete.


In [9]:
X_meta_train_clean = np.column_stack((oof_cat_scores, oof_rcf_scores, oof_incident_entropy))

# FIX (Issue 3 — min_precision raised to 0.80):
# COST_FN raised from 10 → 12 to maintain recall pressure while the higher
# precision floor pushes the threshold upward.  At 80% precision, roughly 1 in
# 5 flagged events may be a false alarm — acceptable for a SOC with automated
# response.  The weight-share check inside train_fusion_meta_learner will warn
# if CatBoost's contribution drops below 30% of total weight, which would
# indicate over-reliance on RCF/entropy and degraded detection of novel attacks.
COST_FN = 12
COST_FP = 2

meta_learner = train_fusion_meta_learner(
    X_meta_train_clean,
    y_bin,
    COST_FN=COST_FN,
    COST_FP=COST_FP,
    lambda_reg=0.1
)

meta_learner.save_model(r"Saves/meta_learner.pkl")

# Compute the optimal decision threshold from OOF scores.
# Done here — on OOF predictions — so the threshold is chosen
# without any information from the held-out test set.
# The result is saved to Saves/optimal_threshold.json so the
# SOAR agent can load it at runtime without re-running training.
oof_meta_scores = meta_learner.predict_proba(X_meta_train_clean)
optimal_threshold = find_optimal_threshold(
    y_bin,
    oof_meta_scores,
    cost_fn=COST_FN,
    cost_fp=COST_FP,
    min_precision=0.75,   # lowered from 0.80 — pairs with min_recall to prevent threshold climbing too high
    min_recall=0.80,      # ensures at least 80% of attacks are caught (guards against high-FN collapse)
    save_path="Saves/optimal_threshold.json"
)



Training Meta-Learner (FN Penalty: 12, FP Penalty: 2, L2: 0.1)...
      ↳ Convergence reached at epoch 1824.

Meta-Learner weights — CatBoost: 3.7395  RCF: 0.2291  Entropy: 0.5021  Bias: 1.9477
CatBoost weight share: 83.6% — healthy contribution.
Meta-Learner Training Complete!
💾 Model state successfully saved to Saves/meta_learner.pkl

--- THRESHOLD OPTIMISATION (OOF) ---
Sweep range    : 0.01 - 0.99 (1000 candidates, 940 skipped due to constraints)
Cost function  : (FN x 12) + (FP x 2)
Constraints    : Precision ≥ 0.75, Recall ≥ 0.80
Optimal threshold    : 0.7801
Precision @ threshold: 0.7505
Recall @ threshold   : 0.9768
Minimum SOC cost     : 42078
  TN=22279  FP=14721  FN=1053  TP=44279
Threshold saved to Saves/optimal_threshold.json


In [10]:
# Train full-dataset base models (no OOF needed here — these are frozen for inference)
catboost_model = train_categorical_model(X_train_cat, y_bin, cat_cols)
incident_model = train_incident_agent(X_train_cat, y_multi, cat_cols)

print("\nSaving Base Models to disk...")
catboost_model.save_model(r"Saves/catboost_base.cbm")
incident_model.save_model(r"Saves/incident_base.cbm")


Training CatBoost (with High Regularization)...

Training Incident Agent (Multi-class Threat Classifier)...

Saving Base Models to disk...


# Testing

In [3]:
print("\n=======================================================")
print("FINAL HOLD-OUT TEST SET EVALUATION")
print("=======================================================\n")

# Load the OOF-optimised threshold and cost constants from disk so this cell
# runs correctly after a kernel restart without needing cells 2-5 in memory.
import json as _json
_threshold_path = "Saves/optimal_threshold.json"
with open(_threshold_path) as _f:
    _threshold_record = _json.load(_f)
optimal_threshold = _threshold_record["optimal_threshold"]
COST_FN = _threshold_record["cost_fn"]
COST_FP = _threshold_record["cost_fp"]
print(f"Loaded threshold: {optimal_threshold:.4f}  (FN×{COST_FN}, FP×{COST_FP})")

test_file_path = r"UNSW-NB15\unsw_test.csv"

# Test mode: prepare_datasets returns X_num already transformed by the saved
# full-train scaler+PCA. X_num_raw is unused in the test path but unpacked
# for API consistency.
X_test_cat, X_test_num, X_test_num_raw, y_test_bin, cat_cols_test, y_test_multi = prepare_datasets(test_file_path, is_train=False)

# Verify PCA dimensionality matches the saved artifact (guards against
# retraining with a different variance threshold or different data)
saved_pca = joblib.load(r"Saves/feature_pca.pkl")
expected_pca_dims = saved_pca.n_components_
assert X_test_num.shape[1] == expected_pca_dims, (
    f"PCA dimensionality mismatch: expected {expected_pca_dims} components, "
    f"got {X_test_num.shape[1]}. Retrain or reload the correct scaler/PCA artifacts."
)

# Load frozen base models
print("\nLoading frozen base models from disk...")
loaded_catboost = CatBoostClassifier()
loaded_catboost.load_model("Saves/catboost_base.cbm")
loaded_rcf = RCF.load_model("Saves/rcf_base.pkl")
loaded_incident = CatBoostClassifier()
loaded_incident.load_model("Saves/incident_base.cbm")

# Generate Level-1 base scores
print("Generating base model scores (CatBoost, RCF, Incident)...")
cat_scores_test = loaded_catboost.predict_proba(X_test_cat)[:, 1]
rcf_scores_test_norm = loaded_rcf.predict_proba(X_test_num)

incident_proba_test = loaded_incident.predict_proba(X_test_cat)
incident_entropy_test = _incident_entropy(incident_proba_test)

# Fuse through meta-learner
print("Fusing scores through the Cost-Sensitive Meta-Learner...")
X_meta_test = np.column_stack((cat_scores_test, rcf_scores_test_norm, incident_entropy_test))
loaded_meta_learner = CostSensitiveMetaLearner.load_model("Saves/meta_learner.pkl")
test_final_risk = loaded_meta_learner.predict_proba(X_meta_test)

# Zero Trust boundary — use the OOF-derived threshold, not 0.5
test_predictions = (test_final_risk >= optimal_threshold).astype(int)

# SOC operational metrics
tn, fp, fn, tp = confusion_matrix(y_test_bin, test_predictions).ravel()
final_soc_cost = (fn * COST_FN) + (fp * COST_FP)

print("\n--- FINAL OPERATIONAL REPORT (TEST SET) ---")
print(f"Decision threshold (OOF-optimised): {optimal_threshold:.4f}")
print(classification_report(y_test_bin, test_predictions, target_names=["Normal (0)", "Attack (1)"]))

print("\n--- ZERO TRUST SOC IMPACT ---")
print(f"True Negatives (Traffic Safely Allowed): {tn}")
print(f"True Positives (Attacks Successfully Blocked): {tp}")
print(f"False Positives (Unjustified Alert Fatigue): {fp}")
print(f"False Negatives (Missed Attacks): {fn}")
print("-------------------------------------------------------")
print(f"Total Operational Penalty Score: {final_soc_cost}")


FINAL HOLD-OUT TEST SET EVALUATION

Loaded threshold: 0.7801  (FN×12, FP×2)
Loading and transforming data from UNSW-NB15\unsw_test.csv (Mode: Test)...
Final Categorical features: 6
Numerical features processed: 20 Principal Components

Loading frozen base models from disk...
RCF state successfully loaded from Saves/rcf_base.pkl
Generating base model scores (CatBoost, RCF, Incident)...


RRCF Blind Test Scoring: 100%|██████████| 175341/175341 [25:01<00:00, 116.82it/s]



[RCF] Score distribution — min=0.0000  mean=0.3720  max=1.0000  % at ceiling (≥0.999): 5.4%
Fusing scores through the Cost-Sensitive Meta-Learner...
Model state successfully loaded from Saves/meta_learner.pkl

--- FINAL OPERATIONAL REPORT (TEST SET) ---
Decision threshold (OOF-optimised): 0.7801
              precision    recall  f1-score   support

  Normal (0)       0.96      0.80      0.87     56000
  Attack (1)       0.91      0.98      0.95    119341

    accuracy                           0.93    175341
   macro avg       0.94      0.89      0.91    175341
weighted avg       0.93      0.93      0.92    175341


--- ZERO TRUST SOC IMPACT ---
True Negatives (Traffic Safely Allowed): 44851
True Positives (Attacks Successfully Blocked): 117352
False Positives (Unjustified Alert Fatigue): 11149
False Negatives (Missed Attacks): 1989
-------------------------------------------------------
Total Operational Penalty Score: 46166


In [11]:
"""
playbook_editor_agent.py — Autonomous Self-Learning Playbook Editor

Implements the autonomous feedback loop described in the SOAR design.
During training/evaluation, if the SOAR engine makes a mistake 
(e.g., isolates ground-truth NORMAL traffic), it triggers this agent:

  [SOAR makes a False Positive during evaluation]
      ↓
  [PlaybookEditorAgent.autonomous_fp_correction()]
      ↓
  [LLM autonomously audits the context and diagnoses the failure]
      ↓
  [LLM synthesises a new routing rule to prevent recurrence]
      ↓
  [Rule + audit record written to playbooks.json]
      ↓
  [dynamic_allowlist updated in agent memory]
      ↓
  [Future events matching this pattern → ALLOW fast-path]
"""

import json
import datetime
import re
import os
import requests
from typing import Optional
import numpy as np

# ---------------------------------------------------------------------------
# Correction record — persisted to correction_log.json
# ---------------------------------------------------------------------------

class CorrectionRecord:
    """
    Immutable record of an autonomous AI self-correction.
    Stored in correction_log.json for audit and batch review.
    """

    def __init__(
        self,
        event_id,
        context: dict,
        ai_decision: dict,
        new_rule: Optional[dict],
        analysis: str,
        corrected_at: str
    ):
        self.event_id     = event_id
        self.context      = context
        self.ai_decision  = ai_decision
        self.new_rule     = new_rule
        self.analysis     = analysis
        self.corrected_at = corrected_at

    def to_dict(self) -> dict:
        return {
            "event_id":     self.event_id,
            "context":      self.context,
            "ai_decision":  self.ai_decision,
            "new_rule":     self.new_rule,
            "analysis":     self.analysis,
            "corrected_at": self.corrected_at
        }


# ---------------------------------------------------------------------------
# PlaybookEditorAgent
# ---------------------------------------------------------------------------

class PlaybookEditorAgent:
    """
    Autonomous self-learning agent that analyses AI mistakes and writes
    new routing rules into playbooks.json so the same mistake is not
    repeated on future events with a matching signature.

    Parameters
    ----------
    soar_agent : ZeroTrustSOARAgent
        The live SOAR agent whose playbooks and memory this editor manages.
    correction_log_file : str
        Path to the JSON file where correction records are persisted.
    ollama_model : str
        Local LLM model name served by Ollama, used for mistake analysis
        and rule synthesis.
    ollama_url : str
        Base URL for the Ollama HTTP API.
    """

    # Maximum routing rules kept in playbooks.json before compaction
    MAX_ROUTING_RULES = 30

    def __init__(
        self,
        soar_agent,
        correction_log_file: str = "correction_log.json",
        ollama_model: str = "deepseek-r1:8b",
        ollama_url: str = "http://localhost:11434/api/generate"
    ):
        self.soar_agent          = soar_agent
        self.correction_log_file = correction_log_file
        self.ollama_model        = ollama_model
        self.ollama_url          = ollama_url

        self.correction_log: list[dict] = self._load_correction_log()

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------

    def autonomous_fp_correction(
        self,
        event_id,
        context: dict,
        decision: dict
    ) -> CorrectionRecord:
        """
        Main entry point — triggered autonomously during evaluation when
        ground-truth Normal traffic is wrongly escalated by the AI.

        Steps
        -----
        1. Extract the behaviour signature from the context.
        2. Ask the LLM to autonomously diagnose WHY the AI made the wrong decision.
        3. Ask the LLM to synthesise a new routing rule that prevents it.
        4. Append the rule to playbooks.json (routing_rules section).
        5. Add the signature to the dynamic allowlist in agent memory.
        6. Persist the correction record to correction_log.json.

        Parameters
        ----------
        event_id    : int | str
        context     : dict  — original context passed to soar_agent.run()
        decision    : dict  — what the AI returned (playbook, reasoning, …)

        Returns
        -------
        CorrectionRecord
        """
        event_id = int(event_id) if hasattr(event_id, "item") else event_id

        print(f"\n{'='*60}")
        print(f"[PlaybookEditorAgent] Triggering Autonomous Self-Healing for Event {event_id}")
        print(f"  AI decided  : {decision.get('playbook')} — {decision.get('reasoning', '')[:80]}")
        print(f"  Ground Truth: NORMAL (False Positive Detected)")

        # ── Step 1: Extract the behaviour signature ───────────────────
        telemetry  = context.get("network_telemetry", {})
        trust_eval_dict = decision.get("trust_eval") or {}
        sig = trust_eval_dict.get("behavior_signature") or self._derive_signature(telemetry)

        print(f"  Behaviour sig: {sig}")

        # ── Step 2: Diagnose the mistake ──────────────────────────────
        analysis = self._analyse_mistake(context, decision)
        print(f"\n[PlaybookEditorAgent] Autonomous Root-cause analysis:\n  {analysis}")

        # ── Step 3: Synthesise a new routing rule ─────────────────────
        new_rule = self._synthesise_rule(context, decision, analysis)
        print(f"\n[PlaybookEditorAgent] Synthesised patch rule:\n  {json.dumps(new_rule, indent=4)}")

        # ── Step 4: Write the rule to playbooks.json ──────────────────
        if new_rule:
            self._append_rule_to_playbooks(new_rule, event_id)

        # ── Step 5: Add signature to dynamic allowlist ────────────────
        self._register_allowlist(sig)

        # ── Step 6: Persist correction record ─────────────────────────
        record = CorrectionRecord(
            event_id     = event_id,
            context      = context,
            ai_decision  = decision,
            new_rule     = new_rule,
            analysis     = analysis,
            corrected_at = datetime.datetime.now().isoformat()
        )
        self.correction_log.append(record.to_dict())
        self._save_correction_log()

        print(f"\n[PlaybookEditorAgent] ✓ Self-Healing complete. "
              f"Playbooks updated. Signature '{sig}' added to allowlist.")
        print(f"{'='*60}\n")

        return record

    # ------------------------------------------------------------------
    # LLM helpers
    # ------------------------------------------------------------------

    def _analyse_mistake(
        self,
        context: dict,
        decision: dict
    ) -> str:
        """
        Asks the local LLM to act as an internal auditor and diagnose
        why the SOAR logic wrongly isolated this ground-truth benign event.
        """
        telemetry   = context.get("network_telemetry", {})
        ml_profile  = context.get("ml_risk_profile", {})
        trust_factors = (decision.get("trust_eval") or {}).get("factors", [])

        system_prompt = """
You are the internal Cognitive Auditor for an AI-driven SOAR system.
The system is currently undergoing evaluation, and you have detected a False Positive:
The SOAR engine escalated and contained an event that is actually GROUND-TRUTH NORMAL benign traffic.

Your task: Explain in 2-4 concise technical sentences WHY the AI made the wrong decision. 
Consider:
  - Which ML signal (fused_risk, categorical_risk, anomaly_risk) was artificially high?
  - Which policy penalties (trust factors) pushed the trust score below threshold?
  - What specific feature of the traffic (proto, service, state, spkts/dpkts) confused the models?
  - Was this a known model hallucination (like CatBoost misclassifying UDP traffic as Shellcode)?

Output ONLY plain text — no JSON, no markdown, no preamble.
"""

        user_prompt = (
            f"AI decision   : {decision.get('playbook')}\n"
            f"AI reasoning  : {decision.get('reasoning', 'none')}\n"
            f"ML profile    : {json.dumps(ml_profile)}\n"
            f"Trust factors : {json.dumps(trust_factors)}\n"
            f"Telemetry     : {json.dumps({k: telemetry.get(k) for k in ['proto','service','state','spkts','dpkts']})}\n"
            f"Threat class  : {context.get('predicted_threat_classification')}\n\n"
            f"Diagnose the root cause of this autonomous False Positive."
        )

        return self._call_llm(system_prompt, user_prompt, max_tokens=1500) or (
            "Unable to generate analysis: LLM unavailable. Autonomous fallback triggered."
        )

    def _synthesise_rule(
        self,
        context: dict,
        decision: dict,
        analysis: str
    ) -> Optional[dict]:
        """
        Asks the local LLM to write a new routing rule that would have
        correctly classified this event.
        """
        telemetry   = context.get("network_telemetry", {})
        ml_profile  = context.get("ml_risk_profile", {})
        threat_type = context.get("predicted_threat_classification", "UNKNOWN")

        existing_rules = self.soar_agent.playbooks.get("routing_rules", [])
        learned_count  = sum(
            1 for r in existing_rules
            if isinstance(r, dict) and r.get("source") == "PlaybookEditorAgent"
        )
        rule_id = f"LEARNED_{learned_count + 1:03d}"

        system_prompt = """
You are the Cognitive Auditor writing a self-healing patch for a SOAR playbook. 
A False Positive occurred, and your job is to write ONE precise routing rule that 
will correctly classify similar benign events as 'ALLOW' in the future.

The rule must be specific enough to avoid over-broad exceptions that would allow real attacks through. 
Anchor it to ≥2 observable signal features (e.g. proto + state + dpkts, or threat_class + fused_risk_range).

Output ONLY a raw JSON object with these exact keys:
{
  "id":        "<LEARNED_NNN>",
  "condition": "IF <specific observable conditions> ...",
  "playbook":  "ALLOW",
  "is_fp":     true,
  "rationale": "<1-2 sentences explaining why this patches the ML hallucination>"
}

No markdown, no preamble, no extra keys.
"""

        user_prompt = (
            f"Rule ID to assign : {rule_id}\n"
            f"Root-cause analysis:\n{analysis}\n\n"
            f"ML profile        : {json.dumps(ml_profile)}\n"
            f"Threat class      : {threat_type}\n"
            f"Key telemetry     : proto={telemetry.get('proto')}, service={telemetry.get('service')}, "
            f"state={telemetry.get('state')}, spkts={telemetry.get('spkts')}, dpkts={telemetry.get('dpkts')}\n\n"
            f"Write the JSON patch rule."
        )

        raw = self._call_llm(system_prompt, user_prompt, max_tokens=1500)
        if not raw:
            return None

        try:
            cleaned = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip()
            for fence in ("```json", "```"):
                if cleaned.startswith(fence):
                    cleaned = cleaned[len(fence):]
            if cleaned.endswith("```"):
                cleaned = cleaned[:-3]
            cleaned = cleaned.strip()

            rule = json.loads(cleaned)
            rule["created_at"] = datetime.datetime.now().isoformat()
            rule["source"]     = "PlaybookEditorAgent"
            return rule

        except (json.JSONDecodeError, ValueError) as exc:
            print(f"[PlaybookEditorAgent] Rule synthesis parse error: {exc}")
            return None

    def _call_llm(self, system_prompt: str, user_prompt: str, max_tokens: int = 400) -> Optional[str]:
        payload = {
            "model":   self.ollama_model,
            "system":  system_prompt,
            "prompt":  user_prompt,
            "stream":  False,
            "options": {
                "temperature": 0.1,
                "num_predict": max_tokens
            }
        }

        try:
            response = requests.post(self.ollama_url, json=payload, timeout=240)
            response.raise_for_status()
            raw = response.json().get("response", "")
            return re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip()
        except requests.exceptions.Timeout:
            print("[PlaybookEditorAgent] LLM timeout during analysis.")
            return None
        except Exception as exc:
            print(f"[PlaybookEditorAgent] LLM call failed: {exc}")
            return None

    # ------------------------------------------------------------------
    # Playbook I/O
    # ------------------------------------------------------------------

    def _append_rule_to_playbooks(self, new_rule: dict, event_id):
        playbooks = self.soar_agent.playbooks

        if "routing_rules" not in playbooks:
            playbooks["routing_rules"] = []
        if "learned_rule_audit" not in playbooks:
            playbooks["learned_rule_audit"] = []

        routing_rules = playbooks["routing_rules"]

        existing_conditions = set()
        for r in routing_rules:
            if isinstance(r, dict):
                existing_conditions.add(r.get("condition", "").strip().lower())

        if new_rule.get("condition", "").strip().lower() in existing_conditions:
            print("[PlaybookEditorAgent] Rule with identical condition already exists — skipping insert.")
            return

        if len(routing_rules) >= self.MAX_ROUTING_RULES:
            self._compact_rules(playbooks)

        # Insert at the TOP of the routing rules (index 0) so the patch overrides older logic
        routing_rules.insert(0, new_rule)

        playbooks["learned_rule_audit"].append({
            "rule_id":      new_rule["id"],
            "event_id":     event_id,
            "trigger":      "Autonomous_Evaluation_FP",
            "appended_at":  new_rule["created_at"]
        })

        self._save_playbooks(playbooks)
        print(f"[PlaybookEditorAgent] Rule '{new_rule['id']}' written to {self.soar_agent.playbook_file}")

    def _compact_rules(self, playbooks: dict):
        routing_rules = playbooks["routing_rules"]
        learned = [r for r in routing_rules if isinstance(r, dict) and r.get("source") == "PlaybookEditorAgent"]
        native  = [r for r in routing_rules if not (isinstance(r, dict) and r.get("source") == "PlaybookEditorAgent")]

        headroom = 5
        keep_count = max(0, self.MAX_ROUTING_RULES - len(native) - headroom)
        learned_trimmed = learned[-keep_count:] if keep_count else []

        playbooks["routing_rules"] = learned_trimmed + native
        print(f"[PlaybookEditorAgent] Compacted routing_rules: kept {len(learned_trimmed)} learned rules.")

    def _save_playbooks(self, playbooks: dict):
        with open(self.soar_agent.playbook_file, "w") as f:
            json.dump(playbooks, f, indent=4)
        self.soar_agent.playbooks = playbooks

    # ------------------------------------------------------------------
    # Memory helpers
    # ------------------------------------------------------------------

    def _register_allowlist(self, sig: str):
        if sig not in self.soar_agent.dynamic_allowlist:
            self.soar_agent.dynamic_allowlist.add(sig)
            self.soar_agent.active_quarantines.discard(sig)
            self.soar_agent._save_memory()
            print(f"[PlaybookEditorAgent] '{sig}' added to dynamic_allowlist (removed from quarantine if present).")

    @staticmethod
    def _derive_signature(telemetry: dict) -> str:
        proto   = telemetry.get("proto",   "unknown")
        service = telemetry.get("service", "unknown")
        state   = telemetry.get("state",   "unknown")
        return f"{proto}_{service}_{state}"

    # ------------------------------------------------------------------
    # Correction log I/O
    # ------------------------------------------------------------------

    def _load_correction_log(self) -> list:
        if not os.path.exists(self.correction_log_file):
            return []
        with open(self.correction_log_file, "r") as f:
            try:
                return json.load(f)
            except json.JSONDecodeError:
                return []

    def _save_correction_log(self):

        def _scrub_numpy(obj):
            if isinstance(obj, dict):
                return {k: _scrub_numpy(v) for k, v in obj.items()}
            if isinstance(obj, list):
                return [_scrub_numpy(v) for v in obj]
            if isinstance(obj, (np.integer,)):
                return int(obj)
            if isinstance(obj, (np.floating,)):
                return float(obj)
            if hasattr(obj, "item"):  # catches pandas + numpy edge cases
                try:
                    return obj.item()
                except:
                    pass
            return obj

        safe_log = _scrub_numpy(self.correction_log)

        with open(self.correction_log_file, "w") as f:
            json.dump(safe_log, f, indent=4)

In [12]:
# -----------------------------------------------------------------------
# CELL 7 — SOAR / Zero Trust Agent Evaluation
#
# Runs a representative sample of test events through the full
# ZeroTrustSOARAgent pipeline:
#   RCF gate → Policy Agent → Zero Trust diamond → Response Agent (LLM)
#   → SOAR Playbook → Update Trust Score → Feedback Loop
#
# All ML scores come from cell 6 — nothing is recomputed here.
# The agent loads its optimal_threshold from Saves/optimal_threshold.json
# automatically, so this cell is safe to run after a kernel restart.
# -----------------------------------------------------------------------


# --- Build the agent (loads threshold, memory, playbooks, policy config) ---
soar_agent = ZeroTrustSOARAgent()

# --- Select a sample of events to evaluate ---
# Evaluate up to N_SAMPLE events: half predicted attacks, half predicted normal
# so we exercise both branches of the ZeroTrust diamond in one run.
N_SAMPLE = 100

# FIX: Randomly sample based on ACTUAL Ground Truth, not predictions.
np.random.seed(42)  # Ensures reproducible results for demonstrations
true_attack_indices = np.where(y_test_bin == 1)[0]
true_normal_indices = np.where(y_test_bin == 0)[0]

sample_indices = np.concatenate([
    np.random.choice(true_attack_indices, N_SAMPLE // 2, replace=False),
    np.random.choice(true_normal_indices, N_SAMPLE // 2, replace=False)
])

# Shuffle so the logs alternate between attacks and normal traffic
np.random.shuffle(sample_indices)
# Predicted threat label per test row (from the incident agent in cell 6)
predicted_threat_labels = loaded_incident.predict(X_test_cat)

print("=" * 60)
print("SOAR / ZERO TRUST AGENT — SAMPLE EVENT EVALUATION")
print("=" * 60)

soar_decisions = []

for i, idx in enumerate(sample_indices):
    print(f"\n{'─' * 60}")
    print(f"Event {i + 1}/{len(sample_indices)}  |  Test row index: {idx}")
    print(f"Ground truth: {'ATTACK' if y_test_bin.iloc[idx] == 1 else 'NORMAL'}")

    # Build the context dict from pre-computed cell-6 scores
    telemetry_row = {
    **X_test_cat.iloc[idx].to_dict(), 
    **X_test_num_raw.iloc[idx].to_dict()  # Use the raw numerics, not the PCA ones
}

    context = soar_agent.construct_context(
        event_id=idx,
        cat_risk=float(cat_scores_test[idx]),
        rcf_risk=float(rcf_scores_test_norm[idx]),
        final_risk=float(test_final_risk[idx]),
        threat_type=str(predicted_threat_labels[idx][0]),
        telemetry=telemetry_row
    )

    # Run the full pipeline — threshold comes from self.optimal_threshold
    decision = soar_agent.run(event_id=idx, context=context)
    soar_decisions.append(decision)

    # -----------------------------------------------------------
    # AUTONOMOUS SELF-HEALING TRIGGER
    # -----------------------------------------------------------
    ground_truth = y_test_bin.iloc[idx]
    playbook = decision.get("playbook")

    # If the traffic was actually Normal, but the AI isolated it
    if ground_truth == 0 and playbook in ["NETWORK_ISOLATION", "STEP_UP_AUTH", "RATE_LIMIT_DOS"]:
        
        # Instantiate the editor if it doesn't exist
        if 'editor_agent' not in locals():
            editor_agent = PlaybookEditorAgent(soar_agent)
            
        # Trigger the autonomous rewrite
        editor_agent.autonomous_fp_correction(
            event_id=idx,
            context=context,
            decision=decision
        )

# --- Summary ---
print(f"\n{'=' * 60}")
print("SAMPLE EVALUATION SUMMARY")
print(f"{'=' * 60}")

playbook_counts = {}
for d in soar_decisions:
    pb = d.get("playbook", "UNKNOWN")
    playbook_counts[pb] = playbook_counts.get(pb, 0) + 1

for playbook, count in sorted(playbook_counts.items()):
    print(f"  {playbook:<22} : {count} event(s)")

fp_overrides = sum(1 for d in soar_decisions if d.get("is_false_positive"))
# FIX: Rename the log to reflect that both the Policy Agent AND the LLM catch False Positives
print(f"\n  Total False-Positive Overrides (Policy + LLM) : {fp_overrides}")
print(f"  Agent memory written to                       : {soar_agent.memory_file}")

[ZeroTrustSOARAgent] Loaded optimal threshold: 0.7801 (OOF SOC cost=42078, FN×12 FP×2)
SOAR / ZERO TRUST AGENT — SAMPLE EVENT EVALUATION

────────────────────────────────────────────────────────────
Event 1/100  |  Test row index: 94554
Ground truth: ATTACK

[ML Gate]   fused_risk=0.9640 > threshold=0.7801 → Escalating to Policy Agent.

[PolicyAgent] Trust Score: 0.0108 | Signature: tcp_ftp_FIN
[ZeroTrust]  Trust UNACCEPTABLE (< 0.4) → Response Agent

[SOAR] Triggering Playbook: NETWORK_ISOLATION for Event 94554
       AI Reasoning: High fused_risk score of 0.964 exceeds the threshold, triggering network isolation. Threat classification as 'Exploits' confirms a potential malicious activity.
       Trust Score: 0.0108 | Factors: 3 | Violations: 0
       ↳ Step 1: Calling VirusTotal_API API -> query_ip_reputation()
       ↳ Step 2: Calling Palo_Alto_Firewall API -> block_src_ip()
       ↳ Step 3: Calling CrowdStrike_EDR API -> isolate_host_network()
       ↳ Step 4: Calling Active_Direct

TypeError: Object of type int64 is not JSON serializable